# COBRA-py SPY Dev Notebook

This notebook sets up a dev workflow, installs/editable package dependencies, downloads SPY OHLCV via yfinance, and runs COBRA-py with both file-based and overridden configs.

## 1. Create dev Workspace and Python Environment

Create a dev folder (if needed), and verify the active interpreter and working paths.

In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd()
project_root = cwd if cwd.name == "rbdpo" else cwd / "rbdpo"
dev_dir = project_root / "dev"
dev_dir.mkdir(parents=True, exist_ok=True)

print("Python executable:", sys.executable)
print("Working directory:", cwd)
print("Project root:", project_root)
print("Dev directory:", dev_dir)
print("PYTHONPATH:", os.environ.get("PYTHONPATH", "<not set>"))

## 2. Install Package in Editable Mode and Add yfinance

Run this once per environment. It keeps local code edits live via editable install.

In [ ]:
# Uncomment and run once if needed:
# !pip install -e ..
# !pip install yfinance

print("Editable install and yfinance are expected to be pre-installed in this environment.")

## 3. Validate Imports and Runtime Paths

In [ ]:
import importlib.metadata as im
import yfinance as yf
import rbdpo

print("cobra-py distribution version:", im.version("cobra-py"))
print("rbdpo package path:", Path(rbdpo.__file__).resolve())
print("yfinance version:", yf.__version__)

## 4. Load Base Configuration from File

In [ ]:
import yaml

config_path = project_root / "configs" / "default.yaml"
with config_path.open("r", encoding="utf-8") as f:
    base_cfg = yaml.safe_load(f)

print("Loaded config from:", config_path)
{
    "objective": base_cfg["objective"]["name"],
    "budget": base_cfg["optimiser"]["budget"],
    "seed": base_cfg["optimiser"]["seed"],
    "train_split": base_cfg["data"]["train_split"],
}

## 5. Define In-Notebook Configuration Overrides

In [ ]:
from copy import deepcopy

def deep_update(base: dict, updates: dict) -> dict:
    out = deepcopy(base)
    for k, v in updates.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = deep_update(out[k], v)
        else:
            out[k] = v
    return out

baseline_override = {
    "objective": {"name": "sharpe", "min_trades": 1},
    "optimiser": {"budget": 20, "seed": 42},
    "indicators": {"n_jobs": 1},
}

alt_override = {
    "objective": {"name": "calmar", "min_trades": 1},
    "optimiser": {"budget": 20, "seed": 123},
    "policy": {"n_entry_rules": 2, "n_exit_rules": 1},
    "indicators": {"n_jobs": 1},
}

cfg_base = deep_update(base_cfg, baseline_override)
cfg_alt = deep_update(base_cfg, alt_override)

{"baseline_objective": cfg_base["objective"]["name"], "alt_objective": cfg_alt["objective"]["name"]}

## 6. Download SPY OHLC Data with yfinance

In [ ]:
import pandas as pd

spy = yf.download("SPY", start="2018-01-01", interval="1d", auto_adjust=False, progress=False)
spy = spy.rename(columns={"Open": "open", "High": "high", "Low": "low", "Close": "close", "Volume": "volume"})
spy.index.name = "datetime"
spy = spy[["open", "high", "low", "close", "volume"]].sort_index()

print("Rows:", len(spy))
print("Has NA:", bool(spy.isna().any().any()))
print("Monotonic index:", bool(spy.index.is_monotonic_increasing))
spy.tail()

## 7. Run Package Flow Using File-Based Config

In [ ]:
from rbdpo.data.loader import load_ohlcv
from rbdpo.data.preprocessor import preprocess
from rbdpo.indicators.precompute import precompute_all
from rbdpo.indicators.registry import DEFAULT_REGISTRY
from rbdpo.search.dehb_runner import run_dehb
from rbdpo.search.space import build_config_space
from rbdpo.reporting.report import generate_report


def run_pipeline(df: pd.DataFrame, cfg: dict, out_dir: Path):
    data = load_ohlcv(df, freq=cfg["data"].get("freq"), min_bars=int(cfg["data"].get("min_bars", 500)))
    train, test = preprocess(data, cfg["data"])

    cache = precompute_all(train, DEFAULT_REGISTRY, n_jobs=int(cfg["indicators"].get("n_jobs", 1)))
    cs = build_config_space(
        n_entry_rules=int(cfg["policy"].get("n_entry_rules", 3)),
        n_exit_rules=int(cfg["policy"].get("n_exit_rules", 1)),
        registry=DEFAULT_REGISTRY,
        seed=int(cfg["optimiser"].get("seed", 42)),
    )

    obj_cfg = {
        "objective": cfg["objective"].get("name", "sharpe"),
        "composite_weights": cfg["objective"].get("composite_weights", [0.5, 0.3, 0.1, 0.1]),
        "complexity_penalty": cfg["objective"].get("complexity_penalty", 0.02),
        "min_trades": cfg["objective"].get("min_trades", 10),
        "n_entry_rules": int(cfg["policy"].get("n_entry_rules", 3)),
        "n_exit_rules": int(cfg["policy"].get("n_exit_rules", 1)),
    }

    result = run_dehb(
        cache=cache,
        data=train,
        config_space=cs,
        obj_config=obj_cfg,
        backtest_config=cfg["backtest"],
        budget=int(cfg["optimiser"].get("budget", 20)),
        seed=int(cfg["optimiser"].get("seed", 42)),
    )

    out_dir.mkdir(parents=True, exist_ok=True)
    report = generate_report(result, wf_result=None, output_path=out_dir)
    return report

base_out = dev_dir / "nb_base_results"
report_base = run_pipeline(spy, base_cfg, base_out)
report_base["summary"]

## 8. Run Package Flow Using Overridden Config

In [ ]:
ovr_out = dev_dir / "nb_override_results"
report_override = run_pipeline(spy, cfg_alt, ovr_out)
report_override["summary"]

## 9. Compare Outputs Across Config Variants

In [ ]:
comparison = pd.DataFrame([
    {
        "run": "base_file_config",
        "objective": base_cfg["objective"]["name"],
        "seed": base_cfg["optimiser"]["seed"],
        "budget": base_cfg["optimiser"]["budget"],
        "best_score": report_base["summary"]["best_score"],
        "evals": report_base["summary"]["n_evaluations"],
    },
    {
        "run": "override_config",
        "objective": cfg_alt["objective"]["name"],
        "seed": cfg_alt["optimiser"]["seed"],
        "budget": cfg_alt["optimiser"]["budget"],
        "best_score": report_override["summary"]["best_score"],
        "evals": report_override["summary"]["n_evaluations"],
    },
])
comparison

## 10. Persist Results and Debug Artifacts in dev/

In [ ]:
effective_dir = dev_dir / "effective_configs"
effective_dir.mkdir(parents=True, exist_ok=True)

with (effective_dir / "base_effective.yaml").open("w", encoding="utf-8") as f:
    yaml.safe_dump(base_cfg, f, sort_keys=False)
with (effective_dir / "override_effective.yaml").open("w", encoding="utf-8") as f:
    yaml.safe_dump(cfg_alt, f, sort_keys=False)

saved = sorted([p.name for p in dev_dir.iterdir()])
print("Saved dev artifacts:")
for name in saved:
    print(" -", name)